## Inspecting and steering a ManifoldEM project

It needs ManifoldEM with its `notebook` extra, installed where the notebook's kernel runs: `pip install "manifoldem[notebook] @ git+https://github.com/flatironinstitute/ManifoldEM"`, or `pip install -e ".[notebook]"` in a checkout.

Set `PROJECT_TOML` in the setup cell to your project's `params_<project>.toml`, the file `manifold-cli init` wrote. The notebook's sections follow the pipeline, and each works as soon as the step it needs has run, so you can come back to it between CLI steps:

| After | Sections |
|---|---|
| `manifold-cli init` | S2 orientation distribution, occupancy thresholds |
| `manifold-cli nlsa-movie` | per-PrD viewers, then anchors and trash PrDs |
| `manifold-cli find-ccs` | CC voting, optical flow |
| `manifold-cli probability-landscape` | trajectory path width (optional) |
| `manifold-cli trajectory` | trajectory probability landscape |

The cells that edit the project (thresholds, anchors, trash PrDs, path width) take over from the old GUI. They save to the project files, so the next CLI step picks the changes up.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Setup — point the notebook at your project and pick a render mode.
#
#   PROJECT_TOML  →  your project's params_<project>.toml, as written by
#                    `manifold-cli init`. A relative path is relative to the
#                    folder this notebook was started in.
#
#   INTERACTIVE = False  →  static plots (Plotly-PNG + matplotlib) that render
#                           on GitHub. Good for sharing this notebook online.
#   INTERACTIVE = True   →  live ipywidgets viewers — clickable S2 sphere, PrD
#                           slider, ψ slider, etc. Use when running locally on
#                           your own data.
# ─────────────────────────────────────────────────────────────────────────────
from ManifoldEM.visualization import load_session

PROJECT_TOML = "path/to/params_<project>.toml"
INTERACTIVE = False

viz = load_session(PROJECT_TOML)

### Plot the S2 tesselation results

### S2 orientation distribution

Points are the **tessellation bin centres** on the unit sphere S2.  
Colour = particle occupancy per bin (number of cryo-EM images whose orientation maps to that bin).  
**Thresholded PDs** (active projection directions after low/high occupancy cuts) are overlaid as larger red markers.

> With `INTERACTIVE=True` the sphere is a clickable Plotly figure (click a red diamond to view its 2D class average and sync the PrD slider below). With `INTERACTIVE=False` you get a static PNG snapshot suitable for GitHub.

In [ ]:
viz.show_s2_sphere(interactive=INTERACTIVE)

### Occupancy thresholds (after `manifold-cli init`)

Every S2 bin with at least `prd_thres_low` particles becomes a projection direction (PrD), and each PrD keeps at most `prd_thres_high` of them. `viz.show_thresholds(low=…, high=…)` previews other values, and `viz.set_thresholds(low, high)` applies them, like `manifold-cli threshold`. New thresholds renumber the PrDs, which clears any anchors and trash PrDs, so every step from `calc-distance` on has to run again.

In [ ]:
viz.show_thresholds(interactive=INTERACTIVE)
# viz.show_thresholds(low=150, high=1000)  # preview other thresholds
# viz.set_thresholds(low=150, high=1000)   # apply them (re-bins the particles)

### Per-PrD eigenvector viewer

**Tab 1** shows the eigenvalue scree plot. **Tab 2** shows the eigenvector movies (`psi_1` … `psi_<num_psi>`).

> With `INTERACTIVE=True`, select a PrD with the slider (or click a diamond on the sphere above). With `INTERACTIVE=False`, the viewer renders PrD 1 (override with `viz.show_prd_viewer(default_prd=N, interactive=INTERACTIVE)`) and the movies are shown as first-frame thumbnails.

In [ ]:
viz.show_prd_viewer(interactive=INTERACTIVE)

### Diffusion map analysis (per PrD)

**Row 1 — Left:** pairwise image distance matrix `D` of the images kept after trimming (`posPath` lists them), with rows and columns sorted by ψ₁. If ψ₁ follows a single continuous change, distances are small near the diagonal and grow away from it.  
**Row 1 — Right:** Ferguson plot — `ln Σ Wᵢⱼ` vs `ln ε` scatter with tanh fit. The inflection gives the optimal kernel width σ; the slope at the inflection estimates the intrinsic dimensionality.  
**Row 2:** ψ₁ vs ψ₂ diffusion map embedding coloured three ways: ψ₃, Riemannian density (`μ`), and in-plane rotation angle (degrees).

> With `INTERACTIVE=True`, this panel is driven by the PrD slider above. With `INTERACTIVE=False` it renders PrD 1 (override via `viz.show_diffusion_map(default_prd=N, interactive=INTERACTIVE)`).

In [ ]:
viz.show_diffusion_map(interactive=INTERACTIVE)

### Anchors and trash PrDs (after `nlsa-movie`, before `find-ccs`)

`find-ccs` starts from *anchor* PrDs whose conformational coordinate you choose by hand: the ψ movie that shows the motion, played forward (`FWD`) or reversed (`REV`). It then propagates that choice along the graph of neighbouring PrDs. Give every connected component at least one anchor, and choose the senses so that all anchors show the same motion (e.g. open → closed). PrDs whose movies look wrong can go in the trash, which leaves them out of `find-ccs` and everything after it.

Watch the movies in the PrD viewer above. PrD and ψ numbers count from 1, as in `topos/PrD_N/psi_K.gif`. Every change is saved to the project immediately.

In [ ]:
viz.show_anchors(interactive=INTERACTIVE)
# viz.set_anchor(prd=1, psi=1, sense="FWD")
# viz.set_anchor(prd=2, psi=3, sense="REV")
# viz.remove_anchor(prd=2)
# viz.trash_prd(4, 5)  # viz.untrash_prd(4, 5) takes them out of the trash again

### CC voting — chosen ψ and FWD/REV sense per PrD

The compilation step votes on (i) which eigenvector ψ_k best captures conformational motion at each PrD and (ii) whether that motion runs in the same (`+1` FWD) or opposite (`−1` REV) sense as a reference PrD. PrDs that get no coordinate (trashed, isolated, or without a usable ψ) are marked ×.

> With `INTERACTIVE=True`, this cell also recolours the red diamonds on the S2 sphere above by their chosen ψ (grey: no coordinate).

In [ ]:
viz.show_cc_voting(interactive=INTERACTIVE)

### Optical-flow quiver per PrD × ψ

For each (PrD, ψ) pair, overlay the forward (`τ: 0→1`) and reverse (`τ: 1→0`) optical-flow vectors on the 2D class average. The chosen ψ (from CC voting) is marked with a ★.

> With `INTERACTIVE=True`, drive this with the PrD and ψ sliders. With `INTERACTIVE=False`, it renders PrD 1 / ψ 1 (override via `viz.show_optical_flow(default_prd=N, default_psi=K, interactive=INTERACTIVE)`).

In [ ]:
viz.show_optical_flow(interactive=INTERACTIVE)

### Trajectory path width (optional, before `manifold-cli trajectory`)

The trajectory step writes one volume per state along the conformational coordinate. With a path width of *w*, each of the `states_per_coord - w + 1` volumes pools the particles of *w* adjacent states. The default is 1.

In [ ]:
# viz.set_path_width(2)

### Trajectory probability landscape

The probability P(τ) of each of the `states_per_coord` states along the conformational coordinate τ (the histogram `hUn`, normalized), and −ln P(τ) relative to its minimum, with the KDE of `tauAvg` overlaid for reference.

In [ ]:
viz.show_trajectory()